### 🔷 What is ETL?

ETL = Extract → Transform → Load

It is a data pipeline process used to move data from source systems into a data warehouse or analytics system.

Used in:

Data Engineering

Data Warehousing

BI & Analytics

Reporting Systems

### 1️⃣ Extract

Collect data from different sources

## Sources can be:

    Databases (MySQL, PostgreSQL)

    APIs

    CSV/Excel files

    Logs

    Cloud storage (S3, GCS)

## Extraction types:

    Full Load

    Incremental Load (CDC – Change Data Capture)

✅ Important concept: Always avoid loading full data daily if not needed.

### 2️⃣ Transform

This is the most important phase.

Typical transformations:

    Data Cleaning (remove nulls, duplicates)

    Standardization (date format, currency format)

    Validation (data type checking)

    Joining multiple datasets

    Aggregations

    Business logic implementation

        Example:
        
        Raw: "2026-03-02T10:00:00Z"
        Transform: "02-03-2026"
### 3️⃣ Load

Load processed data into:

    Data Warehouse (Snowflake, Redshift, BigQuery)
    
    Database
    
    Data Lake

Types:

    Full Load
    
    Incremental Load
    
    Upsert (Insert + Update)

## ETL Architecture

        Source Systems
        
       (DB, API, Files, Logs)
   
              ↓
          Extract Layer
          
              ↓
        Staging Area (Raw Data)
        
              ↓
        Transformation Layer
        
              ↓
        Data Warehouse
        
              ↓
        BI / Reporting / ML
        

### 🔷 Key ETL Concepts To Remember

✅ Idempotency (Running twice should not duplicate data)
✅ Data validation
✅ Logging & Monitoring
✅ Error handling
✅ Retry mechanism
✅ Schema evolution
✅ Performance optimization
✅ Parallel processing
✅ Batch vs Streaming

In [2]:
import pandas as pd

shipments = []
for i in range(1,106):
    shipments.append([
        f"S{i:03}",
        i,
        f"ORD{1000+i}",
        f"Customer{i}",
        "Pune",
        "Mumbai",
        "2026-03-01",
        f"Receiver{i}",
        f"Address Line {i}, City, State, 400{i:03}"
    ])

shipment_df = pd.DataFrame(shipments, columns=[
    "shipment_id","shipment_seq_num","order_number","customer_name",
    "origin","destination","shipment_date",
    "receiver_name","receiver_address"
])

shipment_df.to_csv("shipment.csv", index=False)

In [3]:
import pandas as pd
import random

# Configuration
TOTAL_SHIPMENTS = 105
MAX_PARCELS_PER_SHIPMENT = 3
TOTAL_PARCELS_TARGET = 170

parcels = []
parcel_counter = 1

for shipment_num in range(1, TOTAL_SHIPMENTS + 1):
    
    shipment_id = f"S{shipment_num:03}"
    
    # INTENTIONAL ERROR: Shipment S050 will have no parcels
    if shipment_id == "S050":
        continue
    
    num_parcels = random.randint(1, MAX_PARCELS_PER_SHIPMENT)
    
    for seq in range(1, num_parcels + 1):
        
        if parcel_counter > TOTAL_PARCELS_TARGET:
            break
        
        parcels.append([
            f"P{parcel_counter:03}",                      # parcel_id
            seq,                                          # parcel_seq_num
            f"PO{1000 + parcel_counter}",                 # parcel_order_number
            f"UIDPRC{1000 + parcel_counter}",             # unique_idprc
            shipment_id,                                  # shipment_id
            round(random.uniform(0.5, 5.0), 2),           # weight
            "Created"                                     # status
        ])
        
        parcel_counter += 1

# INTENTIONAL ERROR CASES

# 1️⃣ Invalid shipment reference
parcels.append([
    f"P{parcel_counter:03}",
    1,
    f"PO{1000 + parcel_counter}",
    f"UIDPRC{1000 + parcel_counter}",
    "S200",  # Invalid shipment
    2.5,
    "Created"
])
parcel_counter += 1

# 2️⃣ Zero weight parcel
parcels.append([
    f"P{parcel_counter:03}",
    1,
    f"PO{1000 + parcel_counter}",
    f"UIDPRC{1000 + parcel_counter}",
    "S010",
    0.0,  # Invalid weight
    "Created"
])
parcel_counter += 1

# 3️⃣ Duplicate parcel_seq_num for same shipment
parcels.append([
    f"P{parcel_counter:03}",
    1,  # Duplicate seq for S011
    f"PO{1000 + parcel_counter}",
    f"UIDPRC{1000 + parcel_counter}",
    "S011",
    3.2,
    "Created"
])

# Create DataFrame
parcel_df = pd.DataFrame(parcels, columns=[
    "parcel_id",
    "parcel_seq_num",
    "parcel_order_number",
    "unique_idprc",
    "shipment_id",
    "weight",
    "status"
])

# Save to CSV
parcel_df.to_csv("parcel.csv", index=False)

print("parcel.csv generated successfully with", len(parcel_df), "records")

parcel.csv generated successfully with 173 records


### How To Explain This In Interview (Best Version)


I built an ETL pipeline for a Transport Management System where shipment and parcel data were ingested via API. The pipeline validated shipment-parcel integrity, ensured each shipment had at least one parcel, applied business validations, generated shipping labels for valid shipments, and segregated invalid records into an error table for customer notification. The system ensured idempotency and reliable data processing.

### Project Scenario

We receive:

Shipment details via API

Parcel details via API

We must:

✅ Validate data
✅ Ensure shipment has at least one parcel
✅ Track status
✅ Generate label
✅ Store valid data in DB
✅ Store invalid data in error table
✅ Notify end user using error table

### 🏗️ High-Level ETL Architecture

TMS API (Shipment)  

TMS API (Parcel)

        ↓
     Extract Layer
     
        ↓
     Staging Tables
     
        ↓
     Validation Engine
     
        ↓
  ┌───────────────┬───────────────┐
  
  │ Valid Data    │ Error Data    │
  
  ↓               ↓
  
 Core Tables      Error Table
 
  ↓
  
 Label Generation
 
  ↓
  
 Notification System

## 🗄️ Database Design
1️⃣ shipments
2️⃣ parcels
3️⃣ shipment_status
4️⃣ shipment_labels
5️⃣ error_records

### Sample Input Files
shipment.csv

parcel.csv

### ⚠ Intentional Errors

Shipment S003 → No parcel

Parcel P004 → Invalid shipment S005

Weight could be zero

Missing receiver address

### 🧠 Business Rules

Shipment must have ≥ 1 parcel

Parcel must reference valid shipment

Weight > 0

Receiver address must not be null

Generate label only for valid shipment


### Step 1: Extract

In [5]:
import logging

logging.basicConfig(
    filename="etl.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logging.info("ETL Job Started")

import pandas as pd
import sqlite3
from datetime import datetime

shipment_df = pd.read_csv("shipment.csv")
parcel_df = pd.read_csv("parcel.csv")

### Step 2: Transform & Validation

In [2]:
# import requests
# import time

# def fetch_api_data_with_retry(url, retries=3):
#     for attempt in range(retries):
#         try:
#             response = requests.get(url, timeout=5)
#             response.raise_for_status()
#             return response.json()
#         except Exception as e:
#             logging.warning(f"Attempt {attempt+1} failed: {e}")
#             time.sleep(2)

#     logging.error("API failed after retries")
#     raise Exception("API extraction failed")





error_list = []

# Validate receiver address
invalid_address = shipment_df[shipment_df["receiver_address"].isnull()]

for _, row in invalid_address.iterrows():
    error_list.append({
        "record_type": "SHIPMENT",
        "reference_id": row["shipment_id"],
        "error_message": "Missing receiver address"
    })

shipment_df = shipment_df.dropna(subset=["receiver_address"])

# Validate parcel shipment reference
valid_shipments = set(shipment_df["shipment_id"])
invalid_parcel = parcel_df[~parcel_df["shipment_id"].isin(valid_shipments)]

for _, row in invalid_parcel.iterrows():
    error_list.append({
        "record_type": "PARCEL",
        "reference_id": row["parcel_id"],
        "error_message": "Invalid shipment reference"
    })

parcel_df = parcel_df[parcel_df["shipment_id"].isin(valid_shipments)]

# Validate weight
invalid_weight = parcel_df[parcel_df["weight"] <= 0]

for _, row in invalid_weight.iterrows():
    error_list.append({
        "record_type": "PARCEL",
        "reference_id": row["parcel_id"],
        "error_message": "Invalid weight"
    })

parcel_df = parcel_df[parcel_df["weight"] > 0]

# Validate shipment has at least one parcel
shipment_with_parcel = set(parcel_df["shipment_id"])

for shipment_id in shipment_df["shipment_id"]:
    if shipment_id not in shipment_with_parcel:
        error_list.append({
            "record_type": "SHIPMENT",
            "reference_id": shipment_id,
            "error_message": "Shipment has no parcel"
        })

shipment_df = shipment_df[
    shipment_df["shipment_id"].isin(shipment_with_parcel)
]

### Step 3: Label Generation

In [3]:
label_data = []

for shipment_id in shipment_df["shipment_id"]:
    label_number = f"LBL-{shipment_id}-{datetime.now().strftime('%Y%m%d%H%M%S')}"
    label_data.append({
        "shipment_id": shipment_id,
        "label_number": label_number,
        "generated_at": datetime.now()
    })

label_df = pd.DataFrame(label_data)


### Step 4: Load

In [6]:
conn = sqlite3.connect("tms.db")

shipment_df.to_sql("shipments", conn, if_exists="replace", index=False)
parcel_df.to_sql("parcels", conn, if_exists="replace", index=False)
label_df.to_sql("shipment_labels", conn, if_exists="replace", index=False)

error_df = pd.DataFrame(error_list)
error_df.to_sql("error_records", conn, if_exists="replace", index=False)

conn.close()